[Reference](https://medium.com/@ravishkhullar/okf-rag-the-ultimate-ai-agent-architecture-26b9ceed44f1$0)

In [1]:
from openai import OpenAI
import os
import glob
import yaml

client = OpenAI()

# 1. Load the OKF knowledge bundle from the local directory
def load_okf_bundle(bundle_path: str) -> dict:
    """Reads all Markdown files in the OKF directory into a searchable dict."""
    knowledge = {}
    for filepath in glob.glob(f"{bundle_path}/**/*.md", recursive=True):
        with open(filepath) as f:
            content = f.read()
            # Extract the title from YAML frontmatter
            if content.startswith("---"):
                _, frontmatter, body = content.split("---", 2)
                meta = yaml.safe_load(frontmatter)
                title = meta.get("title", os.path.basename(filepath))
                knowledge[title.lower()] = body.strip()
    return knowledge

# 2. Search OKF (deterministic, keyword-based)
def search_okf(query: str, okf_knowledge: dict) -> str | None:
    """Simple keyword match against OKF titles."""
    for title, content in okf_knowledge.items():
        if title in query.lower():
            return content
    return None

# 3. Search RAG (probabilistic, vector-based)
def search_rag(query: str) -> str:
    """Placeholder for your vector DB search (Pinecone, Weaviate, etc.)."""
    # results = vector_db.similarity_search(query, top_k=5)
    return "RAG context: [retrieved chunks would appear here]"

# 4. The Intelligent Router
def answer_query(query: str, okf_bundle_path: str) -> str:
    okf_knowledge = load_okf_bundle(okf_bundle_path)

    # Try OKF first (deterministic path)
    okf_result = search_okf(query, okf_knowledge)

    if okf_result:
        context = f"[SOURCE: Official Knowledge Base (OKF)]\n{okf_result}"
    else:
        # Fall back to RAG (probabilistic path)
        context = f"[SOURCE: Document Search (RAG)]\n{search_rag(query)}"

    # Send to LLM with the retrieved context
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Answer using ONLY the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ]
    )
    return response.choices[0].message.content